In [ ]:
# Import libraries
import numpy as np
import math
import random
import pandas as pd
import scipy.stats as stats
from statsmodels.stats.multitest import multipletests
import statsmodels.api as sm # linear regression
import seaborn as sns
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import scanpy as sc
import os
import SpatialproxseqClasses as SPC

#*****
mpl.rcdefaults()
# Set font to be arial
mpl.rc('font', **{'sans-serif':'Arial', 'size':12})
mpl.rcParams['mathtext.rm'] = 'sans' 
mpl.rcParams['axes.titlesize'] = 12
# Set default tick size
mpl.rcParams['xtick.major.size'] = 5.5
mpl.rcParams['ytick.major.size'] = 5.5
mpl.rcParams['xtick.minor.size'] = 2.5
mpl.rcParams['ytick.minor.size'] = 2.5
# Default legend settings
mpl.rcParams['legend.fancybox'] = False
mpl.rcParams['legend.edgecolor'] = 'k'

#to store text as text, not as path
new_rc_params = {'text.usetex': False,
                 "svg.fonttype": 'none'}
mpl.rcParams.update(new_rc_params)
#*****
# Seed number
np.random.seed(2025)
random.seed(2025)

In [ ]:
#read_visium data 
adata = sc.read_visium('A1_outs')
adata.var_names_make_unique()
adata.obs['mRNA']= 'RNA'
adata.var["mt"] = adata.var_names.str.startswith("MT-")
sc.pp.calculate_qc_metrics(adata, qc_vars=["mt"], inplace=True)
sc.set_figure_params(figsize=(4, 4))


In [ ]:
print(adata.shape)

In [ ]:
sc.pl.spatial(adata, color = ['total_counts', 'n_genes_by_counts'], size = 1.5, save = 'Fig-1d.svg')

In [ ]:
sc.pl.spatial(adata, color=None, spot_size=0, save = 'Fig-1c.svg')

In [ ]:
# For RNA data:
mean_counts = np.mean(adata.obs['total_counts'])
median_counts = np.median(adata.obs['total_counts'])
std_counts = np.std(adata.obs['total_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")

mean_counts = np.mean(adata.obs['n_genes_by_counts'])
median_counts = np.median(adata.obs['n_genes_by_counts'])
std_counts = np.std(adata.obs['n_genes_by_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")


In [ ]:
#perform UMI based filter
sc.pp.filter_cells(adata, min_counts = 1000)
sc.pp.filter_genes(adata, min_cells=3)

In [ ]:
adata.layers["counts"] = adata.X.copy() # preserve counts
sc.pp.normalize_total(adata, target_sum=1e4) # scale each cell to a common library size
sc.pp.log1p(adata) # log(expression + 1)
adata.raw = adata.copy() # freeze the state in `.raw`
adata.write("adata_A1_rna_raw_data.h5ad")

In [ ]:
sc.pp.highly_variable_genes(
    adata,
    n_top_genes=4000,
    layer="counts",
    flavor="seurat_v3"
)

In [ ]:
sc.pl.highly_variable_genes(adata, log=True)
# subset highly variable genes
adata = adata[:, adata.var.highly_variable].copy()
sc.pp.scale(adata, max_value=10)
sc.tl.pca(adata, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata, log=True)

In [ ]:
sc.pp.neighbors(adata, n_neighbors=15, n_pcs=20) 
sc.tl.umap(adata)

In [ ]:
sc.tl.leiden(adata, resolution = 0.5, key_added='leiden_mrna') 
sc.pl.spatial(adata, color = ['leiden_mrna'],)

In [ ]:
sc.pl.umap(adata, color = ['leiden_mrna'], )

In [ ]:
# Step 1: Subset GC cluster (2 and 3)
adata_gc = adata[adata.obs["leiden_mrna"] .isin(["2", "3"])].copy()

# Step 2: Re-cluster
# restore raw counts
adata_gc.X = adata_gc.layers["counts"].copy()

# store raw counts for dotplot etc.
adata_gc.raw = adata_gc.copy()

# normalization and clustering
sc.pp.normalize_total(adata_gc, target_sum=1e4)
sc.pp.log1p(adata_gc)
sc.pp.highly_variable_genes(adata_gc, n_top_genes=4000)
adata_gc = adata_gc[:, adata_gc.var["highly_variable"]].copy()
sc.pp.scale(adata_gc)
sc.tl.pca(adata_gc)
sc.pp.neighbors(adata_gc)
sc.tl.umap(adata_gc)
sc.tl.leiden(adata_gc, resolution=0.2)

In [ ]:
sc.pl.umap(adata_gc, color = ['leiden'])

In [ ]:
# Step 3: assign back the refined cluster labels to original data
adata.obs.loc[adata.obs["leiden_mrna"].isin(["2", "3"]), "lz_dz_subtype"] = adata_gc.obs["leiden"].values


In [ ]:
mask = adata.obs["leiden_mrna"].isin(["2", "3"])
adata.obs.loc[mask & (adata.obs["lz_dz_subtype"] == "0"), "leiden_mrna"] = "2"
adata.obs.loc[mask & (adata.obs["lz_dz_subtype"] == "1"), "leiden_mrna"] = "3"


In [ ]:
sc.pl.umap(adata, color = ['leiden_mrna'], save = 'Fig-1g-1.svg')

In [ ]:
sc.pl.spatial(adata, color = ['leiden_mrna'], save = 'Fig-1g-2.svg')

In [ ]:
cluster_mapping_mrna = {
    '0': 'Epithelial-crypt',
    '1': 'T:B border',
    '2': 'Light zone',
    '3': 'Dark zone',
    '4': 'Non-GC follicles',
    '5': 'Epithelial-basal cells',
    '6': 'Non-GC follicles', 
    '7': 'Mantle zone',
    '8': 'T cell zone',
}
# Create a new column with the combined cluster name
adata.obs['mrna_annotation'] = adata.obs['leiden_mrna'].map(cluster_mapping_mrna).fillna(adata.obs['leiden_mrna'])

In [ ]:
sc.pl.spatial(adata, color = ['mrna_annotation'],save = 'Fig-1g-3.svg' )


In [ ]:
sc.tl.rank_genes_groups(adata, groupby='mrna_annotation', method='wilcoxon')
sc.pl.rank_genes_groups(adata, groupby='mrna_annotation', method='wilcoxon')

In [ ]:
sc.tl.dendrogram(adata, groupby='mrna_annotation', key_added='dendrogram_annotation')

In [ ]:
sc.pl.rank_genes_groups_heatmap(adata, n_genes=5, groupby='mrna_annotation', swap_axes=True, use_raw=True, 
                                cmap='bwr', figsize=(10, 9))

In [ ]:
sc.pl.umap(adata, color = ['S100A8', 'KRT5', 'SFN', 'S100A2', 'KRT14', 'SPINK5', 'S100A14', 'KRT6A', 'S100A16', 'KRT13', 'mrna_annotation' ],use_raw = True, color_map='coolwarm', ) #basal cells

In [ ]:
sc.pl.dotplot(adata, ['S100A8', 'KRT5', 'SFN', 'S100A2', 'KRT14', 'SPINK5', 'S100A14', 'KRT6A', 'S100A16', 'KRT13',],use_raw = True, groupby="mrna_annotation") #basal

In [ ]:
sc.pl.umap(adata, color = ['ARHGEF16', 'ERRFI1', 'KAZN', 'EPHA2',   'ARHGEF10L', 'PLA2G2F', 'CAMK2N1', 'mrna_annotation'], vmin  = 0, vmax = 1.1, color_map='coolwarm', ) #Surface epithelium

In [ ]:
sc.pl.dotplot(adata, ['ARHGEF16', 'ERRFI1', 'KAZN', 'EPHA2',   'ARHGEF10L', 'PLA2G2F', 'CAMK2N1',], groupby="mrna_annotation") #Surface epithelium

In [ ]:
sc.pl.umap(adata, color = ['CD3D', 'CD3E', 'CD3G', 'CD2', 'IL7R', 'TCF7', 'CCR7', 'LEF1', 'NOSIP', 'GIMAP7', 'mrna_annotation' ], use_raw = True, color_map='coolwarm', ) #CD4 naive

In [ ]:
sc.pl.dotplot(adata, ['CD3D', 'CD3E', 'CD4', 'CD3G', 'CD2', 'IL7R', 'TCF7', 'CCR7', 'LEF1', 'NOSIP', 'GIMAP7',], use_raw = True, groupby="mrna_annotation") #CD4 naive

In [ ]:
sc.pl.umap(adata, color = ['CD83', 'FCER2', 'NME1', 'PHACTR1', 'SYNGR2', 'CCND2', 'DDX21', 'CD72', 'PARP14', 'MIR155HG', 'mrna_annotation' ], color_map='coolwarm', ) #GC-commited NBC

In [ ]:
sc.pl.dotplot(adata, ['CD83', 'FCER2', 'NME1', 'PHACTR1', 'SYNGR2', 'CCND2', 'DDX21', 'CD72', 'PARP14', 'MIR155HG',], swap_axes=True, groupby="mrna_annotation") #GC-commited NBC

In [ ]:
sc.pl.dotplot(adata, ['EGR3', 'CD83', 'PHACTR1', 'CD69', 'IGHD', 'FCER2', 'CD72', 'NFKBID', 'BANK1', 'DUSP2', ], swap_axes=True, groupby="mrna_annotation") #Early GC-commited NBC

In [ ]:
sc.pl.umap(adata, color = ['FCER2', 'IGHD', 'CD83', 'NFKB2', 'CD72', 'CD69', 'MYC', 'EGR3', 'REL', 'HLA-DQB1', 'mrna_annotation' ], color_map='coolwarm', ) #B active marker

In [ ]:
sc.pl.dotplot(adata, ['FCER2', 'IGHD', 'CD83', 'NFKB2', 'CD72', 'CD69', 'MYC', 'EGR3', 'REL', 'HLA-DQB1','CD27'], swap_axes=True, groupby="mrna_annotation") #B active marker

In [ ]:
sc.pl.umap(adata, color = ['IGHD', 'IGHM', 'TCL1A', 'BANK1', 'FCER2', 'HLA-DQB1', 'LINC02397', 'CD72', 'HLA-DQA1', 'HVCN1', 'mrna_annotation' ], color_map='coolwarm', ) #NBC

In [ ]:
sc.pl.dotplot(adata, ['IGHD', 'IGHM', 'TCL1A', 'BANK1', 'FCER2', 'HLA-DQB1', 'LINC02397', 'CD72', 'HLA-DQA1', 'HVCN1',],swap_axes=True, groupby="mrna_annotation") #Naive B cell

In [ ]:
sc.pl.umap(adata, color = ['CD83','LMO2','BCL2A1','mrna_annotation'],  color_map='coolwarm', ) # light zone marker

In [ ]:
sc.pl.dotplot(adata,['CD83','LMO2','BCL2A1',],  groupby="mrna_annotation") # light zone marker

In [ ]:
sc.pl.umap(adata, color = ['CXCR4','AICDA','MME','FOXP1','mrna_annotation'], color_map='coolwarm', ) #dark zone marker

In [ ]:
sc.pl.dotplot(adata,['CXCR4','AICDA','MME','FOXP1',],  groupby="mrna_annotation") # dark zone marker

In [ ]:
cluster_df = pd.DataFrame({
    'cell_index': adata.obs.index,
    'leiden_mrna':adata.obs['leiden_mrna'],
    'mrna_annotation': adata.obs['mrna_annotation'],
})
cluster_df.to_csv('A1_adata_mRNA_cell_clusters.csv', index=False)

In [ ]:
sc.tl.rank_genes_groups(adata, groupby='mrna_annotation', method='wilcoxon')
sc.pl.rank_genes_groups(adata, groupby='mrna_annotation', method='wilcoxon')

In [ ]:
custom_gene_names = [
    'CR1', 'VCAM1', 'IGHM', 'IGHG1', 'CD86', 'PDCD1LG2', 'CR2', 'FCER2', 'IGHD',
    'MS4A1', 'CD19', 'CD40', 'ITGA4', 'ITGB1', 'CD81', 'CD38', 'CD8A', 'CD3E', 
    'CD4', 'CD27', 'PDCD1', 'CD28', 'FCGR2A', 'CD9', 'BSG', 'ICAM1', 'ITGAL', 'CD24'
]


In [ ]:
sc.pl.heatmap(
    adata,
    var_names=custom_gene_names,
    groupby='mrna_annotation',
    swap_axes=True,
    cmap='bwr',
    use_raw=True,
    vmin=-5,
    vmax=5,
    figsize=(10, 8)
)

In [ ]:
#read PLA data and transform the data
data = pd.read_csv('A1-PLA_count_matrix.txt.gz', sep="\t",index_col=0)

In [ ]:
# Update row names (index)
data.index = data.index.str.replace(r"^VCAM1_B:(.*)$", r"CD40_A:\1", regex=True)
data.index = data.index.str.replace(r"^CD29_B:(.*)$", r"CD279_A:\1", regex=True)
data.index = data.index.str.replace(r"^LFA1_B:(.*)$", r"CD11a_A:\1", regex=True)

In [ ]:
valid_condition = ~data.index.str.contains(r'((:.*_A)|(_B:))', regex=True)

# Apply the filter to the dataframe
pla = data[valid_condition]

#remover suffix 
pla.index = pla.index.str.replace(r'(_A|_B)', '', regex=True)

In [ ]:
# #exclude HLADPRQ
pla = pla.loc[~pla.index.str.contains('HLADPRQ'),:]

In [ ]:
# Define the CLR normalization function adapted from Seurat
def clr_function(column):
    # Filter out zeros for log transformation and only include positive values
    positive_values = column[column > 0]
    
    # Calculate the mean of log-transformed positive values
    mean_log = np.sum(np.log1p(positive_values)) / len(column)
    
    # Normalize by subtracting the exponential mean from each log-transformed value
    clr_values = np.log1p(column / np.exp(mean_log))
    
    return clr_values

In [ ]:
pla_obj = SPC.sproxseqObject(pla)
pla_obj.compute_protein_pair_counts()
pla_obj.compute_protein_abundance()


In [ ]:
clr_protein = pla_obj.protein_count.apply(clr_function,axis=0)
clr_protein_pair = pla_obj.protein_pair_count.apply(clr_function,axis=0)

In [ ]:
#read tissue barcode 
tissue_barcode = pd.read_csv('A1_outs/spatial/tissue_positions.csv')
tissue_barcode = tissue_barcode.set_index('barcode')
#keep only PLA spot barcode 
tissue_barcode = tissue_barcode.loc[pla.transpose().index,:]

In [ ]:
#construct pla anndata object
adata_pla = sc.AnnData(pla.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])
adata_protein = sc.AnnData(pla_obj.protein_count.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])
adata_protein_pair = sc.AnnData(pla_obj.protein_pair_count.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])
adata_protein_CLR = sc.AnnData(clr_protein.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])
adata_protein_pair_CLR = sc.AnnData(clr_protein_pair.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])


In [ ]:
#rename var name
adata_pla.var.index = pla.index
adata_protein.var.index = pla_obj.protein_count.index
adata_protein_pair.var.index = pla_obj.protein_pair_count.index
adata_protein_CLR.var.index = pla_obj.protein_count.index
adata_protein_pair_CLR.var.index = pla_obj.protein_pair_count.index


In [ ]:
#read image 
image = mpimg.imread('A1_outs/spatial/tissue_hires_image.png')
def set_uns_parameters(adata, library_id, image):
    spatial_key = "spatial"
    adata.uns[spatial_key] = {library_id: {"images": {"hires": image},
                                           "scalefactors": {'tissue_hires_scalef': 0.5177323,
                                                            'tissue_lowres_scalef': 0.1553197,
                                                            'fiducial_diameter_fullres': 45.467644,
                                                            'spot_diameter_fullres': 28.146637}}}

library_id = "A1_Human_tonsil_Proxseq"
image = image  

set_uns_parameters(adata_pla, library_id, image)
set_uns_parameters(adata_protein, library_id, image)
set_uns_parameters(adata_protein_pair, library_id, image)
set_uns_parameters(adata_protein_CLR, library_id, image)
set_uns_parameters(adata_protein_pair_CLR, library_id, image)

In [ ]:
#CLR normalized protein_pair counts
adata_protein_pair_CLR.obs['PLA'] = 'PLA'
adata_protein_pair_CLR.var['IgG'] = adata_protein_pair_CLR.var_names.str.contains('IgG1a')
adata_protein_pair_CLR.var['feature_types'] = 'protein_pair'
sc.pp.calculate_qc_metrics(
    adata_protein_pair_CLR,
    percent_top=(5, 10, 15),
    var_type="protein_pair",
    qc_vars=("IgG",),
    inplace=True,
)

#keep index both in rna and protein_pair
obs_names = adata_protein_pair_CLR.obs.index.intersection(adata.obs.index)
adata_protein_pair_CLR = adata_protein_pair_CLR[adata_protein_pair_CLR.obs.index.isin(list(obs_names))]
adata = adata[adata.obs.index.isin(list(obs_names))]

In [ ]:
print(adata_protein_pair_CLR.shape)

In [ ]:
adata_protein_pair_CLR.layers["CLR"] = adata_protein_pair_CLR.X.copy() # preserve counts
adata_protein_pair_CLR.raw = adata_protein_pair_CLR.copy() # freeze the state in `.raw`
adata_protein_pair_CLR.write("adata_A1_protein_pair_CLR_data.h5ad")

In [ ]:
sc.pp.scale(adata_protein_pair_CLR, max_value=10)
sc.tl.pca(adata_protein_pair_CLR, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_protein_pair_CLR, log=True)

In [ ]:
sc.pp.neighbors(adata_protein_pair_CLR, n_neighbors=10, n_pcs=11)

In [ ]:
sc.tl.umap(adata_protein_pair_CLR)

In [ ]:
sc.tl.leiden(adata_protein_pair_CLR,key_added='leiden_pla',resolution=0.45)

In [ ]:
sc.pl.spatial(adata_protein_pair_CLR, color='leiden_pla', save = 'Fig-1i-2.svg') 

In [ ]:
sc.pl.umap(adata_protein_pair_CLR, color='leiden_pla',save = 'Fig-1i-1.svg' )

In [ ]:
sc.tl.rank_genes_groups(adata_protein_pair_CLR, groupby='leiden_pla', method='wilcoxon')
sc.pl.rank_genes_groups(adata_protein_pair_CLR, groupby='leiden_pla', method='wilcoxon')

In [ ]:
sc.tl.dendrogram(adata_protein_pair_CLR, groupby='leiden_pla', key_added='dendrogram_annotation')
sc.pl.rank_genes_groups_heatmap(adata_protein_pair_CLR, n_genes=3, groupby='leiden_pla', swap_axes=True, use_raw=False, vmin=-4, vmax=4,
                                cmap='bwr', figsize=(10, 12))

In [ ]:
adata_protein_pair_CLR.obs['mrna_annotation'] = adata.obs['mrna_annotation']
adata_protein_pair_CLR.obs['leiden_mrna'] = adata.obs['leiden_mrna']

In [ ]:
sc.tl.rank_genes_groups(adata_protein_pair_CLR, groupby='mrna_annotation', method='wilcoxon')
sc.pl.rank_genes_groups(adata_protein_pair_CLR, groupby='mrna_annotation', method='wilcoxon')


In [ ]:
sc.tl.dendrogram(adata_protein_pair_CLR, groupby='mrna_annotation', key_added='dendrogram_annotation')
sc.pl.rank_genes_groups_heatmap(adata_protein_pair_CLR, n_genes=4, groupby='mrna_annotation', swap_axes=True, use_raw=False, vmin=-4, vmax=4,
                                cmap='bwr', figsize=(10, 12))

In [ ]:
#raw pla data
adata_protein_pair.obs['PLA'] = 'PLA'
adata_protein_pair.var['IgG'] = adata_protein_pair.var_names.str.contains('IgG1a')
adata_protein_pair.var['feature_types'] = 'protein_pair'
sc.pp.calculate_qc_metrics(
    adata_protein_pair,
    percent_top=(5, 10, 15),
    var_type="protein_pair",
    qc_vars=("IgG",),
    inplace=True,
)



In [ ]:
#keep index both in rna and pla
obs_names = adata_protein_pair.obs.index.intersection(adata.obs.index)
adata_protein_pair = adata_protein_pair[adata_protein_pair.obs.index.isin(list(obs_names))]
adata = adata[adata.obs.index.isin(list(obs_names))]

In [ ]:
print(adata_protein_pair.shape)

In [ ]:
sc.pl.spatial(adata_protein_pair, color = ['total_counts','n_protein_pair_by_counts'],  size = 1.5, save = 'Fig-1e.svg')

In [ ]:

# For PLA data:
mean_counts = np.mean(adata_protein_pair.obs['total_counts'])
median_counts = np.median(adata_protein_pair.obs['total_counts'])
std_counts = np.std(adata_protein_pair.obs['total_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")

mean_counts = np.mean(adata_protein_pair.obs['n_protein_pair_by_counts'])
median_counts = np.median(adata_protein_pair.obs['n_protein_pair_by_counts'])
std_counts = np.std(adata_protein_pair.obs['n_protein_pair_by_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")


In [ ]:
#similar preprocess to rna data
adata_protein_pair.layers["counts"] = adata_protein_pair.X.copy() # preserve counts
adata_protein_pair.raw = adata_protein_pair.copy() # freeze the state in `.raw`
adata_protein_pair.write("adata_A1_protein_pair_raw_data.h5ad")

In [ ]:
#CLR normalized protein data
adata_protein_CLR.obs['protein'] = 'protein'
adata_protein_CLR.var['IgG'] = adata_protein_CLR.var_names.str.contains('IgG1a')
adata_protein_CLR.var['feature_types'] = 'protein'
sc.pp.calculate_qc_metrics(
    adata_protein_CLR,
    percent_top=(5, 10, 15),
    var_type="protein",
    qc_vars=("IgG",),
    inplace=True,
)

#keep index both in rna and protein
obs_names = adata_protein_CLR.obs.index.intersection(adata.obs.index)
adata_protein_CLR = adata_protein_CLR[adata_protein_CLR.obs.index.isin(list(obs_names))]
adata = adata[adata.obs.index.isin(list(obs_names))]

In [ ]:
adata_protein_CLR.layers["CLR"] = adata_protein_CLR.X.copy() # preserve counts
adata_protein_CLR.raw = adata_protein_CLR.copy() # freeze the state in `.raw`
adata_protein_CLR.write("adata_A1_protein_CLR_data.h5ad")

In [ ]:
sc.pp.scale(adata_protein_CLR, max_value=10)
sc.tl.pca(adata_protein_CLR, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_protein_CLR, log=True)

In [ ]:
sc.pp.neighbors(adata_protein_CLR, n_neighbors=20, n_pcs=6)
sc.tl.umap(adata_protein_CLR)

In [ ]:
sc.tl.leiden(adata_protein_CLR,key_added='leiden_protein',resolution=0.8)

In [ ]:
sc.pl.umap(adata_protein_CLR,color='leiden_protein',save = 'Fig-1h-1.svg' ,)

In [ ]:
sc.pl.spatial(adata_protein_CLR, color='leiden_protein',save = 'Fig-1h-2.svg')

In [ ]:
# Copy cluster annotation from `adata` to `adata_protein_CLR`
adata_protein_CLR.obs['mrna_annotation'] = adata.obs['mrna_annotation']
adata_protein_CLR.obs['leiden_mrna'] = adata.obs['leiden_mrna']

In [ ]:
sc.tl.rank_genes_groups(adata_protein_CLR, groupby='mrna_annotation', method='wilcoxon')
sc.pl.rank_genes_groups(adata_protein_CLR, groupby='mrna_annotation', method='wilcoxon')

In [ ]:
sc.tl.dendrogram(adata_protein_CLR, groupby='mrna_annotation', key_added='dendrogram_annotation')

In [ ]:
sc.pl.rank_genes_groups_heatmap(
    adata_protein_CLR,
    groupby='mrna_annotation',
    swap_axes=True,
    cmap='bwr',
    use_raw=False,
    n_genes = 5,
    vmin=-5,
    vmax=5,
    figsize=(9,7)
)


In [ ]:
custom_protein_order = [ 'CD35', 'VCAM1',  'IgM','HIgG', 'CD86','PDL2', 'CD21', 'CD23', 'IgD', 'CD20', 'CD19',  'CD40', 'ITGA4', 'CD24', 'LFA1','CD11a',
    'CD29','CD81','CD38', 'CD8',  'CD3',  'CD4','CD27','CD279','CD28', 'CD32','CD9', 'CD147','ICAM1',]

sc.pl.rank_genes_groups_heatmap(
    adata_protein_CLR,
    groupby='mrna_annotation',
    var_names=custom_protein_order,  
    swap_axes=True,
    cmap='RdBu_r',
    use_raw=False,
    vmin=-3.5,
    vmax=3.5,
    figsize=(6, 6),
    save = 'Fig-2d.svg'
)


In [ ]:
#raw protein data
adata_protein = adata_protein[adata_protein.obs.index.isin(list(obs_names))]
adata_protein.obs['protein'] = 'protein'
adata_protein.var['IgG'] = adata_protein.var_names.str.contains('IgG1a')
adata_protein.var['feature_types'] = 'protein'
sc.pp.calculate_qc_metrics(
    adata_protein,
    percent_top=(5, 10, 15),
    var_type="protein",
    qc_vars=("IgG",),
    inplace=True,
)

In [ ]:
sc.pl.spatial(adata_protein, color = ['total_counts','n_protein_by_counts'],  size = 1.5, save = 'Fig-1f.svg')

In [ ]:
# For protein data:
mean_counts = np.mean(adata_protein.obs['total_counts'])
median_counts = np.median(adata_protein.obs['total_counts'])
std_counts = np.std(adata_protein.obs['total_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")

mean_counts = np.mean(adata_protein.obs['n_protein_by_counts'])
median_counts = np.median(adata_protein.obs['n_protein_by_counts'])
std_counts = np.std(adata_protein.obs['n_protein_by_counts'])

print(f"Mean: {mean_counts:.1f}, Median: {median_counts:.1f}, Std: {std_counts:.1f}")

In [ ]:
adata_protein.layers["counts"] = adata_protein.X.copy() # preserve counts
adata_protein.raw = adata_protein.copy() # freeze the state in `.raw`
adata_protein.write("adata_A1_protein_raw_data.h5ad")

In [ ]:
sc.set_figure_params(figsize=(4, 4))
sc.pl.spatial(adata_protein_CLR, color = ['CD21','CD35','IgD' ], cmap='coolwarm',  use_raw = True, spot_size=45,save = 'Fig-2a-1.svg')

In [ ]:
sc.set_figure_params(figsize=(4, 4))
sc.pl.spatial(adata_protein_CLR, color = ['HIgG','CD3','CD4' ], cmap='coolwarm',  use_raw = True, spot_size=45, save = 'Fig-2a-2.svg')

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_protein_CLR, color = ['ITGA4', ], cmap='coolwarm',  use_raw = True,spot_size=45, save = 'Fig-6a-1.svg')

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_protein_CLR, color = ['CD29', ], cmap='coolwarm',  use_raw = True, spot_size=45, save = 'Fig-6a-2.svg')

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_protein_CLR, color = ['VCAM1', ], cmap='coolwarm',  use_raw = True, spot_size=45, save = 'Fig-6a-3.svg')

In [ ]:
sc.pl.spatial(adata_protein_CLR, color = [ 'CD19', 'CD20', 'CD23','IgM',  'CD38', 'CD40',  'CD24', 'PDL2', 'CD279','CD28','CD86',
   'CD8', 'CD27', 'CD32','CD9','CD81', 'CD147','ICAM1','LFA1','CD11a', 'mIgG1','mIgG2a','mIgG2b'], use_raw = True,
              cmap='coolwarm', spot_size=45,save = 'Extended-Fig-5.svg') #no ITGA4 VCAM1 CD29

In [ ]:
fraction_overlap = pd.read_csv('A1_weighted_sc_pla.csv', index_col=0)
adata_fraction = sc.AnnData(fraction_overlap.transpose().to_numpy(), obsm={'spatial': tissue_barcode[['pxl_col_in_fullres','pxl_row_in_fullres']].to_numpy()}, obs=tissue_barcode[['in_tissue','array_row','array_col']])
adata_fraction.var.index = fraction_overlap.index
set_uns_parameters(adata_fraction, library_id, image)
obs_names = adata_fraction.obs.index.intersection(adata.obs.index)
adata_fraction = adata_fraction[adata_fraction.obs.index.isin(list(obs_names))]
adata_fraction.layers["counts"] = adata_fraction.X.copy() # preserve counts
adata_fraction.raw = adata_fraction.copy() # freeze the state in `.raw`
sc.pp.scale(adata_fraction, max_value=10)
sc.tl.pca(adata_fraction, svd_solver='arpack')
sc.pl.pca_variance_ratio(adata_fraction, log=True)
sc.pp.neighbors(adata_fraction, n_neighbors=10, n_pcs=6)
sc.tl.umap(adata_fraction)
sc.tl.leiden(adata_fraction,key_added='Fractional_overlap',resolution=0.5)
sc.pl.spatial(adata_fraction, color = ['Fractional_overlap'], save='Fig-3c.svg')


In [ ]:
sc.tl.rank_genes_groups(adata_fraction, groupby='Fractional_overlap', method='wilcoxon')
sc.pl.rank_genes_groups(adata_fraction, groupby='Fractional_overlap', method='wilcoxon')
sc.tl.dendrogram(adata_fraction, groupby='Fractional_overlap', key_added='dendrogram_annotation')
sc.pl.rank_genes_groups_heatmap(
    adata_fraction,
    groupby='Fractional_overlap',
    swap_axes=True,
    cmap='bwr',
    use_raw=False,
    n_genes = 4,
    vmin=-5,
    vmax=5,
    figsize=(8,12)
)

In [ ]:
# Copy cluster annotation from `adata` to `adata_fraction`
adata_fraction.obs['mrna_annotation'] = adata.obs['mrna_annotation']
sc.tl.rank_genes_groups(adata_fraction, groupby='mrna_annotation', method='wilcoxon')
sc.pl.rank_genes_groups(adata_fraction, groupby='mrna_annotation', method='wilcoxon')
sc.tl.dendrogram(adata_fraction, groupby='mrna_annotation', key_added='dendrogram_annotation')
sc.pl.rank_genes_groups_heatmap(
    adata_fraction,
    groupby='mrna_annotation',
    swap_axes=True,
    cmap='bwr',
    use_raw=False,
    n_genes = 4,
    vmin=-5,
    vmax=5,
    figsize=(8,10)
)

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_protein_pair_CLR, color = ['ITGA4:VCAM1'],cmap='coolwarm', spot_size=45, layer = 'CLR',save = 'Fig-6d-1.svg', )

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_protein_pair_CLR, color = ['CD29:VCAM1'],cmap='coolwarm', spot_size=45, layer = 'CLR', vmax = 2.3, save = 'Fig-6d-2.svg', )

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_fraction, color = ['CD19:CD21'],cmap='coolwarm', spot_size=45, vmin = 0, vmax = 0.22, save = 'Extended-Fig-13c.svg')
sc.pl.spatial(adata_fraction, color = ['CD21:CD35'],cmap='coolwarm', spot_size=45, vmin = 0.10, vmax = 0.4, save = 'Fig-5c.svg')

In [ ]:
sc.set_figure_params(figsize=(3, 3))
sc.pl.spatial(adata_fraction, color = ['CD20:CD32'],cmap='coolwarm', spot_size=45, vmin = 0, vmax = 0.18, save = 'Fig-5i.svg')


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_pair_box_with_pvalue(
    adata,
    pair,
    groupby="mrna_annotation",
    groups=("Dark zone", "Light zone"),
    layer="CLR",
    palette=("#66c2a5", "#fc8d62"),
    title=None,
    savepath=None,
    dpi=300,
    # fine tuning
    offset_factor=0.10,       # vertical margin above data (as fraction of y-range)
    bar_height_factor=0.04,   # height of the bracket (as fraction of y-range)
    show_p_text=True,         # show exact p-value under the stars
    showfliers=False,         # hide outlier markers from the box
):
    """Compare two groups with a box+strip plot and a robust significance bar."""

    # --- extract values safely
    if layer is not None and layer in adata.layers.keys():
        vals = adata[:, pair].layers[layer]
    elif hasattr(adata, "raw") and adata.raw is not None and pair in adata.raw.var_names:
        vals = adata.raw[:, pair].X
    else:
        vals = adata[:, pair].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.ravel(vals)

    df = pd.DataFrame({pair: vals, "Group": adata.obs[groupby].values})
    df = df[df["Group"].isin(groups)].copy()
    df["Group"] = pd.Categorical(df["Group"], categories=list(groups), ordered=True)

    # --- stats
    g1, g2 = groups
    x1 = df.loc[df["Group"] == g1, pair].values
    x2 = df.loc[df["Group"] == g2, pair].values
    stat, p = mannwhitneyu(x1, x2, alternative="two-sided")

    def p_to_stars(pv):
        return "***" if pv < 1e-3 else "**" if pv < 1e-2 else "*" if pv < 0.05 else "ns"
    stars = p_to_stars(p)

    # --- plot
    plt.figure(figsize=(3.6, 3.2))
    # pass hue to avoid seaborn deprecation; no legend, no dodge
    ax = sns.boxplot(
        data=df, x="Group", y=pair, hue="Group", hue_order=list(groups),
        palette=list(palette), dodge=False, legend=False,
        showcaps=True, showfliers=showfliers,
        boxprops=dict(linewidth=1),
        whiskerprops=dict(linewidth=1),
        medianprops=dict(linewidth=1.4, color="black"),
        saturation=1,
    )
    sns.stripplot(
        data=df, x="Group", y=pair, hue="Group", hue_order=list(groups),
        palette=["black", "black"], dodge=False, legend=False,
        size=2.2, jitter=0.25, alpha=0.35
    )

    # --- robust significance bar placement
    ymin, ymax = ax.get_ylim()
    delta = ymax - ymin
    top = np.nanpercentile(df[pair].values, 99)          # ignore extreme outliers
    y = max(top, ymax) + offset_factor * delta           # vertical position
    h = bar_height_factor * delta                         # bracket height
    ax.set_ylim(ymin, y + (offset_factor + bar_height_factor + 0.05) * delta)

    x_left, x_right = 0, 1
    ax.plot([x_left, x_left, x_right, x_right],
            [y, y + h, y + h, y], lw=1.3, color="black", clip_on=False)
    ax.text((x_left + x_right) / 2, y + h + 0.01 * delta, stars,
            ha="center", va="bottom", fontsize=11, clip_on=False)

    if show_p_text:
        ax.text((x_left + x_right) / 2, y + h - 0.02 * delta,
                f"p = {p:.2e}", ha="center", va="top", fontsize=9, clip_on=False)

    # cosmetics
    sns.despine(offset=6)
    ax.set_xlabel("")
    ax.set_ylabel("Normalized counts")
    ax.set_title(title or pair)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=dpi)
    print(f"Mann–Whitney U p-value: {p:.4g}   ({g1} vs {g2})")
    return ax, p


ax, p = plot_pair_box_with_pvalue(
    adata=adata_protein_pair_CLR,
    pair="ITGA4:VCAM1",
    groupby="mrna_annotation",
    groups=("Dark zone", "Light zone"),
    layer="CLR",
    offset_factor=0.12, bar_height_factor=0.05
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

def plot_pair_box_with_pvalue(
    adata,
    pair,
    groupby="mrna_annotation",
    groups=("Dark zone", "Light zone"),
    layer="CLR",
    palette=("#66c2a5", "#fc8d62"),
    title=None,
    savepath=None,
    dpi=300,
    # fine tuning
    offset_factor=0.10,       # vertical margin above data (as fraction of y-range)
    bar_height_factor=0.04,   # height of the bracket (as fraction of y-range)
    show_p_text=True,         # show exact p-value under the stars
    showfliers=False,         # hide outlier markers from the box
):
    """Compare two groups with a box+strip plot and a robust significance bar."""

    # --- extract values safely
    if layer is not None and layer in adata.layers.keys():
        vals = adata[:, pair].layers[layer]
    elif hasattr(adata, "raw") and adata.raw is not None and pair in adata.raw.var_names:
        vals = adata.raw[:, pair].X
    else:
        vals = adata[:, pair].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.ravel(vals)

    df = pd.DataFrame({pair: vals, "Group": adata.obs[groupby].values})
    df = df[df["Group"].isin(groups)].copy()
    df["Group"] = pd.Categorical(df["Group"], categories=list(groups), ordered=True)

    # --- stats
    g1, g2 = groups
    x1 = df.loc[df["Group"] == g1, pair].values
    x2 = df.loc[df["Group"] == g2, pair].values
    stat, p = mannwhitneyu(x1, x2, alternative="two-sided")

    def p_to_stars(pv):
        return "***" if pv < 1e-3 else "**" if pv < 1e-2 else "*" if pv < 0.05 else "ns"
    stars = p_to_stars(p)

    # --- plot
    plt.figure(figsize=(3.6, 3.2))
    # pass hue to avoid seaborn deprecation; no legend, no dodge
    ax = sns.boxplot(
        data=df, x="Group", y=pair, hue="Group", hue_order=list(groups),
        palette=list(palette), dodge=False, legend=False,
        showcaps=True, showfliers=showfliers,
        boxprops=dict(linewidth=1),
        whiskerprops=dict(linewidth=1),
        medianprops=dict(linewidth=1.4, color="black"),
        saturation=1,
    )
    sns.stripplot(
        data=df, x="Group", y=pair, hue="Group", hue_order=list(groups),
        palette=["black", "black"], dodge=False, legend=False,
        size=2.2, jitter=0.25, alpha=0.35
    )

    # --- robust significance bar placement
    ymin, ymax = ax.get_ylim()
    delta = ymax - ymin
    top = np.nanpercentile(df[pair].values, 99)          # ignore extreme outliers
    y = max(top, ymax) + offset_factor * delta           # vertical position
    h = bar_height_factor * delta                         # bracket height
    ax.set_ylim(ymin, y + (offset_factor + bar_height_factor + 0.05) * delta)

    x_left, x_right = 0, 1
    ax.plot([x_left, x_left, x_right, x_right],
            [y, y + h, y + h, y], lw=1.3, color="black", clip_on=False)
    ax.text((x_left + x_right) / 2, y + h + 0.01 * delta, stars,
            ha="center", va="bottom", fontsize=11, clip_on=False)

    if show_p_text:
        ax.text((x_left + x_right) / 2, y + h - 0.02 * delta,
                f"p = {p:.2e}", ha="center", va="top", fontsize=9, clip_on=False)

    # cosmetics
    sns.despine(offset=6)
    ax.set_xlabel("")
    ax.set_ylabel("Normalized counts")
    ax.set_title(title or pair)
    plt.tight_layout()
    if savepath:
        plt.savefig(savepath, dpi=dpi)
    print(f"Mann–Whitney U p-value: {p:.4g}   ({g1} vs {g2})")
    return ax, p


ax, p = plot_pair_box_with_pvalue(
    adata=adata_protein_pair_CLR,
    pair="CD29:VCAM1",
    groupby="mrna_annotation",
    groups=("Dark zone", "Light zone"),
    layer="CLR",
    offset_factor=0.12, bar_height_factor=0.05
)


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import kruskal, mannwhitneyu
from statsmodels.stats.multitest import multipletests

def plot_pair_box_across_clusters(
    adata,
    pair,
    groupby="mrna_annotation",
    groups=None,
    layer="CLR",
    order="median",
    palette=None,
    title=None,
    savepath=None,
    dpi=300,
    showfliers=False,
    compare_to=None,
    p_adjust="fdr_bh",
    jitter=0.25, dot_size=2.2, dot_alpha=0.35,
    xtick_rotation=60,              # <<< rotate labels
    bottom_margin=0.30,             # <<< extra space for tilted labels
    width_per_category=1.0,         # <<< auto width by #categories
    min_width=5.5, height=3.6
):
    # --- extract vector for `pair`
    if layer is not None and layer in adata.layers.keys():
        vals = adata[:, pair].layers[layer]
    elif hasattr(adata, "raw") and adata.raw is not None and pair in adata.raw.var_names:
        vals = adata.raw[:, pair].X
    else:
        vals = adata[:, pair].X
    if hasattr(vals, "toarray"):
        vals = vals.toarray()
    vals = np.ravel(vals)

    df = pd.DataFrame({pair: vals, "Cluster": adata.obs[groupby].astype(str).values})

    if groups is not None:
        df = df[df["Cluster"].isin(groups)].copy()

    if isinstance(order, (list, tuple)):
        order_list = list(order)
    else:
        cats = df["Cluster"].unique().tolist()
        if order == "alpha":
            order_list = sorted(cats)
        else:
            med = df.groupby("Cluster")[pair].median().sort_values(ascending=False)
            order_list = [c for c in med.index if c in cats]

    df["Cluster"] = pd.Categorical(df["Cluster"], categories=order_list, ordered=True)
    df = df.sort_values("Cluster")

    if palette is None:
        palette = sns.color_palette("Set2", len(order_list))
    elif isinstance(palette, dict):
        palette = [palette[c] for c in order_list]

    # figure size
    fig_w = max(min_width, width_per_category * len(order_list))
    fig, ax = plt.subplots(figsize=(fig_w, height))

    sns.boxplot(
        data=df, x="Cluster", y=pair, hue="Cluster", hue_order=order_list,
        palette=palette, dodge=False, legend=False,
        showcaps=True, showfliers=showfliers,
        boxprops=dict(linewidth=1), whiskerprops=dict(linewidth=1),
        medianprops=dict(linewidth=1.4, color="black"), saturation=1, ax=ax
    )
    sns.stripplot(
        data=df, x="Cluster", y=pair, hue="Cluster", hue_order=order_list,
        palette=["black"]*len(order_list), dodge=False, legend=False,
        size=dot_size, jitter=jitter, alpha=dot_alpha, ax=ax
    )

    ax.set_xlabel("")
    ax.set_ylabel("Normalized counts")
    ax.set_title(title or f"{pair}")
    sns.despine(offset=6)

    # overall KW test
    arrays = [df.loc[df["Cluster"] == c, pair].values for c in order_list]
    H, p_kw = kruskal(*arrays)
    print(f"Kruskal–Wallis across {len(order_list)} clusters: H={H:.3g}, p={p_kw:.3g}")

    # optional pairwise vs reference
    stats_rows = []
    if compare_to is not None and compare_to in order_list:
        ref = compare_to
        p_raw = []
        for c in order_list:
            if c == ref: 
                continue
            x1 = df.loc[df["Cluster"] == ref, pair].values
            x2 = df.loc[df["Cluster"] == c,   pair].values
            _, p = mannwhitneyu(x1, x2, alternative="two-sided")
            stats_rows.append({"ref": ref, "cluster": c, "p": p})
            p_raw.append(p)

        if p_raw:
            rej, p_adj, _, _ = multipletests(p_raw, method=p_adjust)
            for r, padj, rj in zip(stats_rows, p_adj, rej):
                r["p_adj"] = padj
                r["signif"] = "***" if padj < 1e-3 else "**" if padj < 1e-2 else "*" if padj < 0.05 else "ns"

            y0, y1 = ax.get_ylim()
            delta = y1 - y0
            top = np.nanpercentile(df[pair].values, 99)
            base = max(top, y1) + 0.08*delta
            step = 0.06*delta
            for k, r in enumerate(stats_rows):
                i = order_list.index(ref)
                j = order_list.index(r["cluster"])
                x1, x2 = min(i, j), max(i, j)
                y = base + k*step
                ax.plot([x1, x1, x2, x2], [y, y+0.02*delta, y+0.02*delta, y],
                        color="black", lw=1.2, clip_on=False)
                ax.text((x1+x2)/2, y-0.03*delta, r["signif"], ha="center", va="bottom", fontsize=10)
            ax.set_ylim(y0, base + (len(stats_rows)+1)*step)

    # >>> FORCE rotation at the very end (after seaborn/annotations)
    labels = [t.get_text() for t in ax.get_xticklabels()]
    ax.set_xticklabels(labels, rotation=xtick_rotation, ha="right", va="top")
    ax.tick_params(axis="x", pad=2)

    # leave room for the tilted labels
    plt.subplots_adjust(bottom=bottom_margin)

    if savepath:
        plt.savefig(savepath, dpi=dpi, bbox_inches="tight")
    return ax, pd.DataFrame(stats_rows), p_kw


In [ ]:
ax, pairwise_df, p_kw = plot_pair_box_across_clusters(
    adata=adata_protein_pair_CLR,
    pair="ITGA4:VCAM1",
    groupby="mrna_annotation",
    layer="CLR",
    compare_to="Light zone",
    xtick_rotation=45,         
    bottom_margin=0.32,
    min_width=4, 
    width_per_category=0.05,
    savepath = 'Figures/Fig-6g-1.svg',
)


In [ ]:
ax, pairwise_df, p_kw = plot_pair_box_across_clusters(
    adata=adata_protein_pair_CLR,
    pair="CD29:VCAM1",
    groupby="mrna_annotation",
    layer="CLR",
    compare_to="Light zone",
    xtick_rotation=45,        
    bottom_margin=0.32,    
    min_width=4,       
    width_per_category=0.01,
    savepath = 'Figures/Fig-6g-2.svg',)
